# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("J3_Analytics_SQL_Gold")
    .getOrCreate()
)

# TODO : Enregistrer votre table Silver principale sous forme de vue temporaire SQL `v_silver`
SILVER_MAIN_PATH = "/lakehouse/silver/orders_enriched"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").collect()[0][0])

## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [ ]:
# Résumé statistique de deux métriques continues : le délai de livraison (jours) et la valeur de la commande (R$)
METRICS = ["delivery_time_days", "order_total_value"]

df_dist_summary = None
for metric in METRICS:
    part = spark.sql(f"""
        SELECT
            '{metric}'                                                   AS metrique,
            COUNT(*)                                                     AS n,
            ROUND(AVG({metric}), 2)                                      AS moyenne,
            ROUND(STDDEV({metric}), 2)                                   AS ecart_type,
            ROUND(SKEWNESS({metric}), 2)                                 AS skewness,
            CAST(PERCENTILE_APPROX({metric}, 0.25) AS DOUBLE)            AS p25,
            CAST(PERCENTILE_APPROX({metric}, 0.50) AS DOUBLE)            AS p50_mediane,
            CAST(PERCENTILE_APPROX({metric}, 0.75) AS DOUBLE)            AS p75,
            CAST(PERCENTILE_APPROX({metric}, 0.90) AS DOUBLE)            AS p90,
            CAST(PERCENTILE_APPROX({metric}, 0.95) AS DOUBLE)            AS p95,
            CAST(MAX({metric}) AS DOUBLE)                                AS max_val
        FROM v_silver
    """)
    df_dist_summary = part if df_dist_summary is None else df_dist_summary.unionByName(part)

assert df_dist_summary is not None, "TODO : Écrivez la requête SQL de résumé statistique."
df_dist_summary.show(truncate=False)


**Votre justification statistique :**
> **Délai de livraison (`delivery_time_days`)** : moyenne = 12,5 jours, médiane = 10 jours, écart-type = 9,56, **skewness = 3,83**, P90 = 23 j, P95 = 29 j, maximum = 210 jours.
> **Valeur de la commande (`order_total_value`)** : moyenne = 159,83 R$, médiane = 105,28 R$, écart-type = 218,80, **skewness = 9,37**, P95 = 445,64 R$, maximum = 13 664 R$.
>
> Les deux distributions sont **très asymétriques à droite** (skewness largement supérieur à 0, donc queue étirée vers les grandes valeurs). La moyenne est tirée vers le haut par quelques valeurs extrêmes : elle dépasse la médiane de 2,5 jours pour les délais (+25 %) et de 54,55 R$ pour les montants (+52 %). Un client « typique » est donc livré en 10 jours et dépense environ 105 R$, pas 12,5 jours et 160 R$. Le délai maximal de 210 jours et la commande à 13 664 R$ sont des cas isolés qui n'ont presque aucun poids dans la médiane.
>
> **Indicateurs retenus pour le dashboard** : la **médiane** (P50) comme valeur centrale et l'**IQR** (P75 − P25 : 9 jours pour les délais, 114,39 R$ pour les montants) comme mesure de dispersion, complétés par le **P90** pour mesurer les cas dégradés (1 commande sur 10 prend plus de 23 jours). La moyenne reste affichée à titre indicatif, notamment pour le chiffre d'affaires total, où l'additivité est justement ce qu'on cherche. Ce choix implique aussi, au J4, des tests **non paramétriques** (Mann-Whitney, Kruskal-Wallis, Spearman), car la condition de normalité n'est pas respectée.


In [ ]:
GOLD_PARETO_PATH = "/lakehouse/gold/revenue_by_category"

df_gold_1 = spark.sql("""
    WITH by_category AS (
        SELECT
            main_category,
            COUNT(*)                                                 AS nb_orders,
            ROUND(SUM(order_total_value), 2)                         AS revenue_total,
            PERCENTILE_APPROX(order_total_value, 0.50)               AS median_order_value,
            PERCENTILE_APPROX(order_total_value, 0.75)
                - PERCENTILE_APPROX(order_total_value, 0.25)         AS iqr_order_value,
            ROUND(100 * AVG(CAST(is_late AS INT)), 2)                AS late_rate_pct,
            ROUND(AVG(review_score), 2)                              AS avg_review_score
        FROM v_silver
        GROUP BY main_category
    )
    SELECT
        main_category,
        nb_orders,
        revenue_total,
        median_order_value,
        iqr_order_value,
        late_rate_pct,
        avg_review_score,
        DENSE_RANK() OVER (ORDER BY revenue_total DESC)              AS revenue_rank,
        ROUND(100 * SUM(revenue_total) OVER (
                ORDER BY revenue_total DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
              ) / SUM(revenue_total) OVER (), 2)                     AS cumul_pct
    FROM by_category
    ORDER BY revenue_rank
""")
assert df_gold_1 is not None, "TODO : Écrivez la requête SQL avec fenêtre de Pareto dans `df_gold_1`."

(
    df_gold_1.coalesce(1)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_PARETO_PATH)
)
df_gold_1.show(15, truncate=False)
n_pareto = df_gold_1.filter("cumul_pct <= 80").count() + 1
print(f"{n_pareto} catégories sur {df_gold_1.count()} réalisent environ 80 % du chiffre d'affaires.")


## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [ ]:
GOLD_TIME_PATH = "/lakehouse/gold/monthly_performance"

# 2016 ne compte que quelques commandes : un taux de croissance sur cette base serait trompeur
df_gold_2 = spark.sql("""
    WITH monthly AS (
        SELECT
            purchase_month,
            COUNT(*)                                                 AS nb_orders,
            ROUND(SUM(order_total_value), 2)                         AS revenue_total,
            ROUND(AVG(delivery_time_days), 2)                        AS avg_delivery_days,
            PERCENTILE_APPROX(delivery_time_days, 0.50)              AS median_delivery_days,
            ROUND(100 * AVG(CAST(is_late AS INT)), 2)                AS late_rate_pct,
            ROUND(AVG(review_score), 2)                              AS avg_review_score
        FROM v_silver
        WHERE purchase_month >= '2017-01'
        GROUP BY purchase_month
    )
    SELECT
        *,
        LAG(revenue_total, 1) OVER (ORDER BY purchase_month)         AS prev_month_revenue,
        ROUND(100 * (revenue_total / LAG(revenue_total, 1) OVER (ORDER BY purchase_month) - 1), 2)
                                                                     AS revenue_growth_pct,
        ROUND(AVG(revenue_total) OVER (
                ORDER BY purchase_month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2)
                                                                     AS revenue_moving_avg_3m
    FROM monthly
    ORDER BY purchase_month
""")
assert df_gold_2 is not None, "TODO : Écrivez la requête SQL temporelle avec LAG() dans `df_gold_2`."

(
    df_gold_2.coalesce(1)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_TIME_PATH)
)
df_gold_2.show(30, truncate=False)


## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'` et expliquez pourquoi la multiplication de micro-fichiers pénalise le NameNode HDFS.

**Pourquoi les micro-fichiers pénalisent le NameNode HDFS ?**
Le NameNode garde en **mémoire vive** les métadonnées de chaque fichier et de chaque bloc (environ 150 octets par objet), quelle que soit la taille réelle du fichier. Un million de petits fichiers de quelques Ko occupe donc autant de métadonnées qu'un million de gros fichiers, alors qu'ils ne contiennent presque rien : le NameNode sature en mémoire avant que les disques ne soient pleins. S'y ajoutent des lectures plus lentes (une tâche Spark par petit fichier, beaucoup d'ouvertures et d'appels au NameNode pour peu de données lues) et des rapports de blocs plus lourds depuis les DataNodes. Chaque écriture Delta ajoute des fichiers Parquet : `OPTIMIZE` les regroupe en fichiers plus gros. Sur notre table Silver, il a fusionné **10 fichiers en 1** (≈ 14 Mo, sous la taille de bloc de 16 Mo). C'est aussi pour cela que les tables Gold sont écrites avec `.coalesce(1)`.


In [ ]:
# 3e table Gold : indicateurs logistiques par État du client + catégorie la plus rentable de chaque État
GOLD_STATE_PATH = "/lakehouse/gold/state_logistics"

df_gold_3 = spark.sql("""
    WITH state_category AS (
        SELECT customer_state, main_category, SUM(order_total_value) AS category_revenue
        FROM v_silver
        GROUP BY customer_state, main_category
    ),
    top_category AS (
        SELECT customer_state, main_category AS top_category, ROUND(category_revenue, 2) AS top_category_revenue
        FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_state ORDER BY category_revenue DESC) AS rn
            FROM state_category
        )
        WHERE rn = 1
    ),
    state_kpi AS (
        SELECT
            customer_state,
            COUNT(*)                                                 AS nb_orders,
            ROUND(SUM(order_total_value), 2)                         AS revenue_total,
            PERCENTILE_APPROX(delivery_time_days, 0.50)              AS median_delivery_days,
            ROUND(100 * AVG(CAST(is_late AS INT)), 2)                AS late_rate_pct,
            ROUND(AVG(review_score), 2)                              AS avg_review_score
        FROM v_silver
        GROUP BY customer_state
    )
    SELECT k.*, t.top_category, t.top_category_revenue
    FROM state_kpi k
    JOIN top_category t ON k.customer_state = t.customer_state
    ORDER BY k.revenue_total DESC
""")

(
    df_gold_3.coalesce(1)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_STATE_PATH)
)
df_gold_3.show(30, truncate=False)

spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").show(truncate=False)
